In [1]:
import numpy as np
import joblib
import tensorflow as tf

In [2]:
class GenomicClassifier:
    def __init__(self, model_path, vectorizer_path):
        self.model = tf.keras.models.load_model(model_path)
        self.cv = joblib.load(vectorizer_path)

    def _one_hot_encode_seqs(self, sequences, max_len=500):
      mapping = {"A": 0, "C": 1, "G": 2, "T": 3}
      encoded_arr = np.zeros((len(sequences), max_len, 4))

      for i, seq in enumerate(sequences):
          for j, base in enumerate(seq[:max_len]): # Truncate/Pad
              if base in mapping:
                  encoded_arr[i, j, mapping[base]] = 1
      return encoded_arr
    def _get_kmers(self,sequences, k=5):
      return [" ".join([seq[i:i+k] for i in range(len(seq)-k+1)]) for seq in sequences]
    def _map_ids_to_classes(self, dom_id, spec_id):
      species_map = {
          0: "Arabidopsis thaliana",
          1: "Bacillus subtilis",
          2: "Bos taurus",
          3: "Escherichia coli",
          4: "Glycine max",
          5: "Homo sapiens",
          6: "Mus musculus",
          7: "Oryza sativa",
          8: "Pseudomonas aeruginosa",
          9: "Rattus norvegicus",
          10: "Staphylococcus aureus",
          11: "Zea mays"
      }

      domain_map = {
          0: "Bacteria",
          1: "Metazoa",
          2: "Viridiplantae"
      }

      domain_name = domain_map.get(dom_id, "Unknown Domain")
      species_name = species_map.get(spec_id, "Unknown Species")

      return domain_name, species_name
    def predict_sequence(self, sequence):

        test_seq = [sequence]
        x_dna = self._one_hot_encode_seqs(test_seq)
        x_km = self.cv.transform(self._get_kmers(test_seq)).toarray()


        raw_preds = self.model.predict([x_dna, x_km])


        dom_id = np.argmax(raw_preds[0], axis=1)[0]
        spec_id = np.argmax(raw_preds[1], axis=1)[0]


        dom_conf = np.max(raw_preds[0]) * 100
        spec_conf = np.max(raw_preds[1]) * 100


        domain_name, species_name = self._map_ids_to_classes(dom_id, spec_id)

        return {
            "domain_name": domain_name,
            "species_name": species_name,
            "domain_confidence": dom_conf,
            "species_confidence": spec_conf
        }

In [3]:
pip install streamlit biopython pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 56.7 MB/s eta 0:00:00


In [4]:
%%writefile app.py
import streamlit as st
import pandas as pd
from Bio import SeqIO
import io
import numpy as np
import joblib
import tensorflow as tf


st.set_page_config(
    page_title="Genomic Classifier AI",
    page_icon="🧬",
    layout="centered"
)


class GenomicClassifier:
    def __init__(self, model_path, vectorizer_path):
        self.model = tf.keras.models.load_model(model_path)
        self.cv = joblib.load(vectorizer_path)

    def _one_hot_encode_seqs(self, sequences, max_len=500):
        mapping = {"A": 0, "C": 1, "G": 2, "T": 3}
        encoded_arr = np.zeros((len(sequences), max_len, 4))

        for i, seq in enumerate(sequences):
            for j, base in enumerate(seq[:max_len]): # Truncate/Pad
                if base in mapping:
                    encoded_arr[i, j, mapping[base]] = 1
        return encoded_arr

    def _get_kmers(self, sequences, k=5):
        return [" ".join([seq[i:i+k] for i in range(len(seq)-k+1)]) for seq in sequences]

    def _map_ids_to_classes(self, dom_id, spec_id):
        species_map = {
            0: "Arabidopsis thaliana", 1: "Bacillus subtilis", 2: "Bos taurus",
            3: "Escherichia coli", 4: "Glycine max", 5: "Homo sapiens",
            6: "Mus musculus", 7: "Oryza sativa", 8: "Pseudomonas aeruginosa",
            9: "Rattus norvegicus", 10: "Staphylococcus aureus", 11: "Zea mays"
        }
        domain_map = {0: "Bacteria", 1: "Metazoa", 2: "Viridiplantae"}

        return domain_map.get(dom_id, "Unknown Domain"), species_map.get(spec_id, "Unknown Species")

    def predict_sequence(self, sequence):
        test_seq = [sequence]
        x_dna = self._one_hot_encode_seqs(test_seq)
        x_km = self.cv.transform(self._get_kmers(test_seq)).toarray()

        raw_preds = self.model.predict([x_dna, x_km])

        dom_id = np.argmax(raw_preds[0], axis=1)[0]
        spec_id = np.argmax(raw_preds[1], axis=1)[0]

        dom_conf = np.max(raw_preds[0]) * 100
        spec_conf = np.max(raw_preds[1]) * 100

        domain_name, species_name = self._map_ids_to_classes(dom_id, spec_id)

        return {
            "domain_name": domain_name,
            "species_name": species_name,
            "domain_confidence": dom_conf,
            "species_confidence": spec_conf
        }


@st.cache_resource
def load_real_classifier():
    MODEL_PATH = "genomic_hybrid_83pct.keras"
    VECTORIZER_PATH = "k5_vectorizer.joblib"

    try:
        return GenomicClassifier(MODEL_PATH, VECTORIZER_PATH)
    except Exception as e:
        st.error(f"Failed to load model from paths. Internal Error: {e}")
        return None


classifier = load_real_classifier()


def process_fasta_file_content(file_content, classifier_instance):
    results = []
    fasta_file_io = io.StringIO(file_content.decode('utf-8'))

    try:
        for record in SeqIO.parse(fasta_file_io, "fasta"):
            sequence_id = record.id
            sequence = str(record.seq).upper().replace('N', 'A')
            expected_seq_len = 500

            if len(sequence) < expected_seq_len:
                st.warning(f"⚠️ **{sequence_id}**: Length ({len(sequence)} bp) is less than {expected_seq_len} bp. Skipping.")
                continue


            if not sequence:
                continue

            prediction_output = classifier_instance.predict_sequence(sequence)
            results.append({
                "Sequence ID": sequence_id,
                "Predicted Domain": prediction_output["domain_name"],
                "Domain Confidence": prediction_output['domain_confidence'],
                "Predicted Species": prediction_output["species_name"],
                "Species Confidence": prediction_output['species_confidence']
            })
    except Exception as e:
        st.error(f"Error processing sequences: {e}")
        return pd.DataFrame()

    return pd.DataFrame(results)


st.title("🧬 Genomic Classifier Platform")
st.markdown("Upload **FASTA** or **TXT** files to run real-time deep learning taxonomic classification.")

if classifier is None:
    st.error("🛑 App waiting for valid model and vectorizer file paths. Check file locations in Colab.")
else:
    uploaded_files = st.file_uploader("Choose FASTA or TXT file(s)", type=["fasta", "fa", "txt"], accept_multiple_files=True)

    if uploaded_files:
        all_results_dfs = []

        if st.button("🚀 Run Genomic Analysis", type="primary"):
            with st.spinner("Deep learning model running inference... Please wait."):
                for uploaded_file in uploaded_files:
                    file_content = uploaded_file.read()
                    st.subheader(f"Analyzing: `{uploaded_file.name}`")

                    df_results = process_fasta_file_content(file_content, classifier)
                    if not df_results.empty:
                        all_results_dfs.append(df_results)

            if all_results_dfs:
                final_df = pd.concat(all_results_dfs, ignore_index=True)
                st.success("🎉 Analysis Complete!")

                st.dataframe(final_df.style.format({
                    "Domain Confidence": "{:.2f}%",
                    "Species Confidence": "{:.2f}%"
                }), use_container_width=True)

                csv = final_df.to_csv(index=False).encode('utf-8')
                st.download_button(label="📥 Download Results as CSV", data=csv, file_name="genomic_predictions.csv", mime="text/csv")
            else:
                st.error("No sequences were successfully validated.")

Writing app.py


In [5]:
!pip install pyngrok -q

In [6]:
import subprocess
from pyngrok import ngrok


NGROK_TOKEN = ""
!ngrok config add-authtoken {NGROK_TOKEN}


subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])


try:
    for tunnel in ngrok.get_tunnels():
        ngrok.disconnect(tunnel.public_url)
except Exception as e:
    print(f"Warning: Could not disconnect existing tunnels: {e}")


ngrok.kill()


public_url = ngrok.connect(8501, proto="http")

print(f" SUCCESS! Click the link below to open your stable app:")
print(f" {public_url.public_url}")


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
 SUCCESS! Click the link below to open your stable app:
 https://unilingual-tanisha-dissidently.ngrok-free.dev


In [ ]:

!pkill -f ngrok

